### Introduction
The aim of this notebook to demonstrate interactions with various dataclasses.
Currently each test case is comprised of the following:
- Base case / prompt
- Full history of presenting complaint
- Full medical history - systems review
- Past medical history
- Examination
- Differentials to consider
- Key investigations and findings
- Further differentials
- Key management plans - NB: This can be further investigation as well

In [1]:
# TODO: Make a GUI to make it super simple to generate a new case

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%env OPENAI_API_KEY=""
                   
if not 'dirguard' in locals():
  %cd ..
  dirguard = True

%aimport vivabench
%aimport vivabench.util
%aimport vivabench.ontology.concepts

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: OPENAI_API_KEY=""


In [3]:
from vivabench.ontology import concepts
from vivabench.ontology.concepts import *

In [4]:
# Base classes such as HOPC, Differential require specifications
brief_history = "A 32 year old woman presents with 3 day history of abdominal pain"
full_history = "A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding."
hopc = HistoryOfPresentingComplaint(brief_history, full_history)

print(hopc.prompt())

BRIEF HISTORY: A 32 year old woman presents with 3 day history of abdominal pain
FULL HISTORY: A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding.



In [5]:
# Composite classes have defaults
pmh = PastMedicalHistory()
print(pmh.prompt())

pmh = PastMedicalHistory(medical_history="Appendectomy 2 weeks ago")
print(pmh.prompt())

MEDICAL HISTORY: No significant medical history
SURGICAL HISTORY: No significant surgical history
ALLERGIES: No known drug allergies
FAMILY HISTORY: Parents both healthy. No significant family history
SOCIAL HISTORY: Works as an office worker. Non-smoker

MEDICAL HISTORY: Appendectomy 2 weeks ago
SURGICAL HISTORY: No significant surgical history
ALLERGIES: No known drug allergies
FAMILY HISTORY: Parents both healthy. No significant family history
SOCIAL HISTORY: Works as an office worker. Non-smoker



In [6]:
# Investigations have default values
ix =  Investigation.get_default("sodium")
print(ix.prompt())

# So do sets of investigations 
vitals = InvestigationSet.get_default("vitals")
print(InvestigationSet.from_dict(vitals.as_dict()).prompt()) # NB: Testing as_dict and from_dict methods

Sodium: 140 mmol/L

VITALS
- Temperature: 36.5 °C
- Pulse: 60 beats/minute
- Blood Pressure: 110/70 mm Hg
- Respiratory Rate: 12 / minute




In [7]:
# Most dataclasses can be outputted as dictionary for storage with .as_dict() or as prompt for agent with .prompt()
pe = PhysicalExamination()
pe.as_dict()

{'vitals': {'name': 'vitals',
  'investigations': {'temperature': {'full_name': 'Temperature',
    'value': '36.5',
    'unit': '°C',
    'type': 'vitals'},
   'pulse': {'full_name': 'Pulse',
    'value': '60',
    'unit': 'beats/minute',
    'type': 'vitals'},
   'blood_pressure': {'full_name': 'Blood Pressure',
    'value': '110/70',
    'unit': 'mm Hg',
    'type': 'vitals'},
   'respiratory_rate': {'full_name': 'Respiratory Rate',
    'value': '12',
    'unit': '/ minute',
    'type': 'vitals'}}},
 'skin': 'Warm to touch. No rash or cyanosis.',
 'heent': 'Moist mucous membrane.',
 'pulmonary': 'Breath sounds are equal bilaterally with good air movement in all fields. No wheezing.',
 'cardiovascular': 'Regular rate and rhythm; no murmurs, rubs, or gallop. No peripheral edema.',
 'gastrointestinal': 'Bowel sounds normal. Abdomen soft, non-tender.',
 'genitourinary': 'Unremarkable',
 'musculoskeletal': 'Unremarkable',
 'neurological': 'Unremarkable',
 'mental_status': 'Alert and orien

In [8]:
print(pe.prompt())

VITALS
- Temperature: 36.5 °C
- Pulse: 60 beats/minute
- Blood Pressure: 110/70 mm Hg
- Respiratory Rate: 12 / minute

SKIN: Warm to touch. No rash or cyanosis.
HEENT: Moist mucous membrane.
PULMONARY: Breath sounds are equal bilaterally with good air movement in all fields. No wheezing.
CARDIOVASCULAR: Regular rate and rhythm; no murmurs, rubs, or gallop. No peripheral edema.
GASTROINTESTINAL: Bowel sounds normal. Abdomen soft, non-tender.
GENITOURINARY: Unremarkable
MUSCULOSKELETAL: Unremarkable
NEUROLOGICAL: Unremarkable
MENTAL_STATUS: Alert and oriented



In [11]:
# Similarly, we have differential sets and management sets
ddx_1 = Differential("Surgical bleeding")
ddx_2 = Differential("Recurrent appendicitis")
ddx = DifferentialList(values=[ddx_1, ddx_2])
print(ddx.prompt())

- Surgical bleeding
- Recurrent appendicitis




In [12]:
mx_1 = Management("Exploratory Laparaotomy")
mx_2 = Management("Ceftriaxone", "1", "g", route="IV", frequency="stat")
mx = ManagementList(values=[mx_1, mx_2])
print(mx.prompt())

- Exploratory Laparaotomy   
- Ceftriaxone 1g IV stat




In [20]:
# Debug - Output .json
case_dict = ClinicalCase(hopc, SystemsReview(), pmh, pe, ddx,{'vitals': vitals},  ddx, ddx_1, mx).as_dict()
case = ClinicalCase.from_dict(case_dict)
case.as_dict()

{'hopc': {'brief_history': 'A 32 year old woman presents with 3 day history of abdominal pain',
  'full_history': 'A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding.'},
 'history': {'skin': 'Noncontributory',
  'heent': 'Noncontributory',
  'pulmonary': 'Noncontributory',
  'cardiovascular': 'Noncontributory',
  'gastrointestinal': 'Noncontributory',
  'genitourinary': 'Noncontributory',
  'musculoskeletal': 'Noncontributory',
  'neurological': 'Noncontributory',
  'mental_status': 'Noncontributory'},
 'pmh'

In [19]:
print(case.prompt())

== HISTORY OF PRESENTING COMPLAINT
BRIEF HISTORY: A 32 year old woman presents with 3 day history of abdominal pain
FULL HISTORY: A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding.

== SYSTEMS REVIEW
SKIN: Noncontributory
HEENT: Noncontributory
PULMONARY: Noncontributory
CARDIOVASCULAR: Noncontributory
GASTROINTESTINAL: Noncontributory
GENITOURINARY: Noncontributory
MUSCULOSKELETAL: Noncontributory
NEUROLOGICAL: Noncontributory
MENTAL STATUS: Noncontributory

== PAST MEDICAL HISTORY
MEDICAL HISTORY: Appendec